# **TRAINING SCRIPT**
## *Chess Neural Network*

### **I - SETUP PHASE**

#### 1. Importing libraries

In [1]:
import torch
import pickle
import numpy as np

from tqdm import tqdm
from pathlib import Path
from model import ChessCNN
from torch.utils.data import DataLoader, TensorDataset

#### 2. Defining global variables

On essaie de forcer les calculs effectués lors de l'entrainement sur le GPU en priorité. Ensuite nous initialisons les variables d'entrainement :
- Une taille de batch à 64 - *le modèle ne voit que 64 positions par 64 positions*
- Un nombre d'epochs à 100 - *l'entrainement va faire 100 passages sur le dataset complet*
- Un learning rate (taux d'apprentissage) à 0.0001 - *à chaque batch le réseau se rend compte de ses erreurs et va légèrement corriger*  

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-4

### **II - LOADING PHASE**

#### 1. Loading datasets

On charge en mémoire les données préparées par le script `Scripts/dataset.py`.

- `X.npy` : Il prend la forme d'un tableau NumPy (N, 13, 8, 8) où chaque élément est une position d'échecs encodée
- `y.npy` : Il prend la forme d'un tableau NumPy (N) où chaque valeur valeur est l'indice d'un coup joué en partie (stocké dans X)

En sortant les données de cette manière nos faisons en sorte que le modèle ne prédit pas les coups 'texte' mais les indices des coups.

Enfin `num_moves` recense le nombre total de coups uniques dans le dataset et va définir la taille de la dernière couche du réseau.

In [3]:
meta = np.load("../Data/Processed Database/X_meta.npy")
shape = tuple(meta)

X = np.memmap("../Data/Processed Database/X.dat", dtype=np.float32, mode="r", shape=shape)
y = np.load("../Data/Processed Database/y.npy")

with open("../Data/Processed Database/move_to_int.pkl", "rb") as f:
    move_to_int = pickle.load(f)

num_moves = len(move_to_int)

dataset = TensorDataset(torch.tensor(X),
                        torch.tensor(y))

#### 2. Loading model & optimizers

In [4]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = ChessCNN(num_moves).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### **III - TRAINING PHASE**

On entraîne le modèle sur `EPOCHS` avec reprise automatique depuis un checkpoint (si disponible).
 
- **Reprise depuis un checkpoint**

Au démarrage, on cherche le checkpoint le plus récent dans `Models/Checkpoints` et on le restaure. 

- **Boucle d'entraînement**

À chaque batch on effectue le cycle classique : *`zero_grad` → `forward` → `loss` → `backward` → `optimizer.step`*.

- **Checkpoints & sauvegarde finale**

Un checkpoint est sauvegardé à la fin de chaque epoch. Il contient le modèle, l'optimizer et le numéro de la prochaine epoch (*epoch + 1*).

In [ ]:
CHECKPOINT_DIR = Path("../Models/Checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

existing = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pth"))

if existing:

    latest = existing[-1]
    checkpoint = torch.load(latest, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    start_epoch = checkpoint["epoch"]
    print(f"Reprise depuis {latest.name} (epoch {start_epoch})")

else:
    start_epoch = 0
    print("Entraînement depuis zéro")

for epoch in range(start_epoch, EPOCHS):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for xb, yb in loop:

        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)

        loss = criterion(logits, yb)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)

        correct += (preds == yb).sum().item()
        total += yb.size(0)

        loop.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100 * correct / total:.2f}%")

    avg_loss = total_loss / len(loader)
    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | avg_loss={avg_loss:.4f} | acc={accuracy:.2f}%")

    checkpoint = {"epoch": epoch + 1, 
                  "model_state": model.state_dict(),
                  "optimizer_state": optimizer.state_dict()}
    
    torch.save(checkpoint, CHECKPOINT_DIR / f"checkpoint_epoch_{epoch+1:03d}.pth")

torch.save(model.state_dict(), f"../Models/mdl_{BATCH_SIZE}_{EPOCHS}_{LR}.pth")
print("Model saved")

C:\Users\thoma\AppData\Local\Temp\ipykernel_15224\4214239463.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(latest, map_location=DEVICE)


Reprise depuis checkpoint_epoch_034.pth (epoch 34)


Epoch 35/100: 100%|██████████| 62473/62473 [11:36<00:00, 89.72it/s, acc=41.53%, loss=1.5459] 


Epoch 35/100 | avg_loss=2.0021 | acc=41.53%


Epoch 36/100: 100%|██████████| 62473/62473 [09:03<00:00, 115.05it/s, acc=41.61%, loss=1.6463]


...

Epoch 49/100: 100%|██████████| 62473/62473 [08:52<00:00, 117.40it/s, acc=42.31%, loss=2.0109]


Epoch 49/100 | avg_loss=1.9666 | acc=42.31%


Epoch 50/100: 100%|██████████| 62473/62473 [08:06<00:00, 128.33it/s, acc=42.35%, loss=2.2471]


Epoch 50/100 | avg_loss=1.9651 | acc=42.35%


Epoch 51/100:  88%|████████▊ | 54806/62473 [07:48<01:26, 88.40it/s, acc=42.45%, loss=1.9328] 